In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import BinaryType

tables = ["Customers","Products","Employees","Orders",
          "Order_Details","Categories","Suppliers"]

def load(table):
    path = f"Files/raw/northwind/{table}"
    names = [f.name.lower() for f in notebookutils.fs.ls(path)]
    has_parquet = any(n.endswith(".parquet") for n in names)

    if has_parquet:                                   # OData source -> Parquet
        df = spark.read.parquet(path)
    else:                                             # REST source -> raw JSON wrapper
        raw = spark.read.option("multiline", "true").json(f"{path}/*.json")
        df = raw.select(F.explode("value").alias("r")).select("r.*")

    # drop paging/metadata columns and any binary image columns
    junk = [c for c in df.columns if c.startswith("@odata")]
    binary = [f.name for f in df.schema.fields if isinstance(f.dataType, BinaryType)]
    return df.drop(*junk, *binary, "Photo", "Picture")

StatementMeta(, 42a022be-92d2-4e34-8bf8-4668bc8a3e32, 3, Finished, Available, Finished, False)

In [2]:
for t in tables:
    df = load(t)
    print(t, df.count())
    df.printSchema()

StatementMeta(, 42a022be-92d2-4e34-8bf8-4668bc8a3e32, 4, Finished, Available, Finished, False)

Customers 91
root
 |-- CustomerID: string (nullable = true)
 |-- CompanyName: string (nullable = true)
 |-- ContactName: string (nullable = true)
 |-- ContactTitle: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- PostalCode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Fax: string (nullable = true)

Products 77
root
 |-- ProductID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- SupplierID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- QuantityPerUnit: string (nullable = true)
 |-- UnitPrice: decimal(38,18) (nullable = true)
 |-- UnitsInStock: short (nullable = true)
 |-- UnitsOnOrder: short (nullable = true)
 |-- ReorderLevel: short (nullable = true)
 |-- Discontinued: boolean (nullable = true)

Employees 9
root
 |-- EmployeeID: integer (nullable = true)
 |-- LastName: string (n

In [3]:
for t in tables:
    df = load(t).dropDuplicates()
    df.write.mode("overwrite").option("overwriteSchema", "true") \
      .format("delta").saveAsTable(f"silver_{t.lower()}")
    print("wrote", f"silver_{t.lower()}", df.count())

StatementMeta(, 42a022be-92d2-4e34-8bf8-4668bc8a3e32, 5, Finished, Available, Finished, False)

wrote silver_customers 91
wrote silver_products 77
wrote silver_employees 9
wrote silver_orders 830
wrote silver_order_details 2155
wrote silver_categories 8
wrote silver_suppliers 29
